In [ ]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.tools import Tool
from IPython.display import display, Markdown
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost, GrpcWorkerAgentRuntime
from dotenv import load_dotenv

load_dotenv(override=True)


In [ ]:
@dataclass
class Message:
    content: str

In [ ]:
host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

In [ ]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [ ]:
instruction1 = "You are to support a motion in a debate on whether a junior developer should use Coding Agent."

instruction2 = "You are to oppose a motion in a debate on whether a junior developer should use Coding Agent."

judge = "You must make a decision on whether to use Coding Agent for a project. \
Your research team has come up with the following reasons for and against. \
You must make a decision based on the following reasons for and against and select one of the two sides that wins the debate."

In [ ]:
class AgainstAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class SupportAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("support", "default")
        inner_2 = AgentId("against", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Support of Debate:\n{response1.content}\n\n## Against Debate:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [ ]:

worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
await worker1.start()
await SupportAgent.register(worker1, "support", lambda: SupportAgent("support"))

worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
await worker2.start()
await AgainstAgent.register(worker2, "against", lambda: AgainstAgent("against"))

worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
await worker.start()
await Judge.register(worker, "judge", lambda: Judge("judge"))
agent_id = AgentId("judge", "default")

In [ ]:
response = await worker.send_message(Message(content="Go!"), agent_id)

display(Markdown(response.content))

await worker.stop() 
await worker1.stop()
await worker2.stop()

await host.stop()